# Basic fitting in **EasyDynamics**
We here show how to use **EasyDynamics** to fit a more advanced synthetic data set with multiple signals and a background, as well as an energy offset that could from from slight instrument misalignment. 

The general procedure is to create an `Experiment` object to hold the data, a `SampleModel` to describe the model, and `Analysis` to fit the model to the data. The `Analysis` object now also contains an `InstrumentModel`, which describes the background and energy offset.

In [1]:
# Imports
import pooch

import easydynamics as edyn

# Make the plots interactive
%matplotlib widget

As before, we create an `Experiment` to hold the data, and rebin it.

In [2]:
# Load the data
experiment = edyn.Experiment(display_name='Tutorial')

file_path = pooch.retrieve(
    url='https://github.com/easyscience/dynamics-lib/raw/refs/heads/master/docs/docs/tutorials/data/fake_advanced_data.hdf5',
    known_hash='818202c758ae074df1f1848e6571f9640757c6f034c2b77d25cdc16edd68f359',
)

experiment.load_hdf5(filename=file_path)
experiment.rebin({'Q': 16, 'energy': 256})
experiment.plot_data(slicer=True)

WidgetFigure(children=(HBar(), HBar(children=(VBar(children=(Toolbar(children=(ButtonTool(icon='home', layout=…

This time, we see a sharp Gaussian near zero energy transfer, but with broad tails. There is also a pair of symmetrically placed peaks near +-1.5 meV. The background is not zero, and looks like it has a slope. We first focus on the three contributions to the signal: the Gaussian, its tails, which can be described by a Lorentzian, and the peaks near +- 1.5 meV, which can be described by a damped harmonic oscillator.

We create and place these in a `ComponentCollection` like this:

In [3]:
gaussian = edyn.Gaussian(name='Gaussian', area=3, width=0.05)
lorentzian = edyn.Lorentzian(name='Lorentzian', area=2, width=0.3)
dho = edyn.DampedHarmonicOscillator(name='DHO', area=1.5, width=0.2, center=1.5)

collection = edyn.ComponentCollection()
collection.append_component(gaussian)
collection.append_component(lorentzian)
collection.append_component(dho)

<details>
  <summary><strong>Note</strong></summary>
<div style="border-left: 4px solid #2196F3; background:#e3f2fd; padding:10px;">
Note that we do not give the Gaussian and Lorentzian a `center` value. In this case, the center is at zero (with a potential offset, see below), and fixed. This is typically what you want when describing elastic (the Gaussian) and quasielastic (the Lorentzian) signals.
</div>
</details>
We pass this `ComponentCollection` to our `SampleModel`, just like before when we passed only a single component.

In [4]:
model = edyn.SampleModel(components=collection)

Next, we want to handle the two instrument effects. The first is the small shift in energy: if you zoom in on the peak at the center, you'll see that it is in fact not centered at zero - it is offset. The offset applies to elastic, inelastic and quasielastic scattering. This is very typical in real experiments, and can be caused by slight misalignment or imperfections in the instrument. To handle this, we create an `InstrumentModel`, and give it an `energy_offset`. 

<details>
  <summary><strong>Note</strong></summary>
<div style="border-left: 4px solid #2196F3; background:#e3f2fd; padding:10px;">
As noted in the previous tutorial, the energy_offset is always present, even when you do not explicitly set an InstrumentModel. This may give problems in two cases: 1) If all peaks are given a center value and allowed to fit it. In the simplest case of a single peak, this is problematic because there are now two parameters determining the center of the peak. The peak center/energy offset are equivalent in this case, and cannot be uniquely determined. 2) If there is no central peak, e.g. if only background scattering is present.
In either case, the energy_offset should be fixed. You can do this in the InstrumentModel using InstrumentModel.fix_energy_offset() or Analysis using Analysis.fix_energy_offset().
</div>
</details>

In [5]:
instrument = edyn.InstrumentModel(energy_offset=0.05)

Second, we want to handle the background. We do this with a `BackgroundModel`, which works the same as the `SampleModel`. We give it a `Polynomial` component:
<details>
  <summary><strong>💡 Tip</strong></summary>
  <div style="padding:10px; margin-top:5px; border-left:4px solid #4caf50; background:#e8f5e9;">
    SampleModel, BackgroundModel and ResolutionModel (introduced later) all take components in several different ways: you can add a single component like in the previous tutorial, append components using the `append_component` method, or give a ComponentCollection like we do here. 
  </div>
</details>

<details>
  <summary><strong>💡 Tip</strong></summary>
  <div style="padding:10px; margin-top:5px; border-left:4px solid #4caf50; background:#e8f5e9;">
    Polynomials will warn if they produce negative values, since negative backgrounds are unphysical. To suppress the warning, set suppress_warnings to True, either when constructing the Polynomial (polynomial=sm.Polynomial(coefficients=[0.0], suppress_warnings=True)), or afterwards (polynomial.suppress_warnings = True).
  </div>
</details>


In [6]:
background = edyn.BackgroundModel(components=edyn.Polynomial(coefficients=[1.2, 0.05]))

We add the background to the instrument model:
<details>
  <summary><strong>💡 Tip</strong></summary>
  <div style="padding:10px; margin-top:5px; border-left:4px solid #4caf50; background:#e8f5e9;">
    We could also have created the background model first, and then added it to the InstrumentModel when creating it, i.e. InstrumentModel(energy_offset = 0.05, background_model = background)
  </div>
</details>


In [7]:
instrument.background_model = background

Now we create out `Analysis` object and give it the instrument model:

In [8]:
analysis = edyn.Analysis(
    experiment=experiment,
    sample_model=model,
    instrument_model=instrument,
)

It is always good to check if the model somewhat matches the data.

In [9]:
analysis.plot_data_and_model()

WidgetFigure(children=(HBar(), HBar(children=(VBar(children=(Toolbar(children=(ButtonTool(icon='home', layout=…

As before, let us first fit a single $Q$ index and plot the data and model to see how it looks. We choose an arbitrary $Q$ and plot only that one. We also plot the residuals underneath by setting plot_residuals to True:

In [10]:
fit_result_independent_single_Q = analysis.fit(Q_index=5)
analysis.plot_data_and_model(Q_index=5, plot_residuals=True)

WidgetFigure(children=(HBar(), HBar(children=(VBar(children=(Toolbar(children=(ButtonTool(icon='home', layout=…

The fit looks very good. We can again get a list of the parameters for this fit by accesing the corresponding `Analysis1d` object. Note that the Gaussian and Lorentzian centers are both zero, but that the `energy_offset` is non-zero.

In [11]:
analysis.analysis_list[5].get_all_parameters()

[<Parameter 'Gaussian area': 3.5330 ± 0.0222 meV, bounds=[0.0:inf]>,
 <Parameter 'Gaussian center': 0.0000 meV (fixed), bounds=[-inf:inf]>,
 <Parameter 'Gaussian width': 0.0499 ± 0.0002 meV, bounds=[1e-10:inf]>,
 <Parameter 'Lorentzian area': 10.4445 ± 0.0739 meV, bounds=[0.0:inf]>,
 <Parameter 'Lorentzian center': 0.0000 meV (fixed), bounds=[-inf:inf]>,
 <Parameter 'Lorentzian width': 0.3887 ± 0.0046 meV, bounds=[1e-10:inf]>,
 <Parameter 'DHO area': 2.5689 ± 0.0405 meV, bounds=[0.0:inf]>,
 <Parameter 'DHO center': 1.4854 ± 0.0012 meV, bounds=[1e-10:inf]>,
 <Parameter 'DHO width': 0.1013 ± 0.0020 meV, bounds=[1e-10:inf]>,
 <Parameter 'energy_offset': 0.1216 ± 0.0002 meV, bounds=[-inf:inf]>,
 <Parameter 'instrument_scale': 1.0000 (fixed), bounds=[0.0:inf]>,
 <Parameter 'Polynomial_c0': 1.1892 ± 0.0159, bounds=[-inf:inf]>,
 <Parameter 'Polynomial_c1': 0.2072 ± 0.0049, bounds=[-inf:inf]>]

Since the fit looked good, we can now fit all $Q$. We also plot the result, again using the slicer.

In [12]:
fit_result_all_Q = analysis.fit()
analysis.plot_data_and_model(plot_residuals=True, autoscale=False)

WidgetFigure(children=(HBar(), HBar(children=(VBar(children=(Toolbar(children=(ButtonTool(icon='home', layout=…

Let us plot a few parameters as a function of `Q` using the `plot_parameters` method.

In [13]:
analysis.plot_parameters(names=['Gaussian area', 'DHO area', 'DHO center'])

WidgetFigure(children=(HBar(), HBar(children=(VBar(children=(Toolbar(children=(ButtonTool(icon='home', layout=…

With apologies for the lack of creativity, these all appear like straight lines. We can fit them individually or all together using `ParameterAnalysis`.

For each parameter we want to fit, we create a fit function (here a `Polynomial`) and a `FitBinding` connecting the parameter name to the fit function. We then pass all bindings to a single `ParameterAnalysis` object:

<details>
  <summary><strong>Note</strong></summary>
<div style="border-left: 4px solid #2196F3; background:#e3f2fd; padding:10px;">
Two units must be set on each fit function:

- `x_unit='1/angstrom'` — because `ParameterAnalysis` always uses Q as its x-axis.
- `y_unit='meV'` — because all three parameters (`Gaussian area`, `DHO area`, `DHO center`) have unit `meV`.
</div>
</details>

In [14]:
gauss_fit_func = edyn.Polynomial(
    coefficients=[3.7, -0.5], x_unit='1/angstrom', y_unit='meV', name='Gauss area fit'
)
dho_area_fit_func = edyn.Polynomial(
    coefficients=[2.0, 0.12], x_unit='1/angstrom', y_unit='meV', name='DHO area fit'
)
dho_center_fit_func = edyn.Polynomial(
    coefficients=[1.1, 0.2], x_unit='1/angstrom', y_unit='meV', name='DHO center fit'
)

binding1 = edyn.FitBinding(model=gauss_fit_func, targets='Gaussian area')

binding2 = edyn.FitBinding(model=dho_area_fit_func, targets='DHO area')

binding3 = edyn.FitBinding(model=dho_center_fit_func, targets='DHO center')

parameter_analysis = edyn.ParameterAnalysis(
    parameters=analysis,
    bindings=[binding1, binding2, binding3],
)

parameter_analysis.plot()

WidgetFigure(children=(HBar(), HBar(children=(VBar(children=(Toolbar(children=(ButtonTool(icon='home', layout=…

The start guesses look reasonable, so we fit and plot the result:

In [15]:
parameter_analysis.fit()
parameter_analysis.plot()

WidgetFigure(children=(HBar(), HBar(children=(VBar(children=(Toolbar(children=(ButtonTool(icon='home', layout=…